In [ ]:
"""Day 25 / Session 1 - Lab Assignment 2: Self- vs Cross-Attention & a Future-Leakage Audit."""
import numpy as np


def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - np.max(z, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)


def attention(Q, K, V, mask=None):
    Q = np.asarray(Q, dtype=float)
    K = np.asarray(K, dtype=float)
    V = np.asarray(V, dtype=float)
    scores = Q @ K.T / np.sqrt(Q.shape[-1])
    if mask is not None:
        scores = scores + mask                               # additive mask: -inf blocks a position
    weights = softmax(scores, axis=-1)
    return weights @ V, weights


def self_attention(X, Wq, Wk, Wv, causal=False):
    X = np.asarray(X, dtype=float)
    Q, K, V = X @ Wq, X @ Wk, X @ Wv                         # Q, K and V all come from the SAME sequence
    mask = None
    if causal:
        n = X.shape[0]
        mask = np.zeros((n, n))
        mask[np.triu_indices(n, k=1)] = -np.inf
    return attention(Q, K, V, mask)


def cross_attention(X_dec, X_enc, Wq, Wk, Wv):
    Q = np.asarray(X_dec, dtype=float) @ Wq                  # queries from the decoder
    K = np.asarray(X_enc, dtype=float) @ Wk                  # keys and values from the encoder
    V = np.asarray(X_enc, dtype=float) @ Wv
    return attention(Q, K, V, None)


def future_leakage(weights):
    w = np.asarray(weights, dtype=float)
    if w.shape[0] < 2:
        return 0.0
    return float(np.max(np.triu(w, k=1)))                    # biggest weight on a future position
